# 📈 trend_following — daily long/short trend starts, ridden behind a ratcheting ATR stop

**The idea.** A trend **starts** when the close breaks the prior `entry_window`-bar high (long) or low (short) *and* Wilder's ADX ≥ `adx_min` with ±DI agreeing — the price left its range, and the move is directional. The book has `max_positions` (20) equal slots; when more names start than slots are free, the **strongest 12-1 month relative trend** goes first. The **stop** starts `atr_mult` ATRs from the signal close and then trails as a **chandelier** — the best price since entry minus `atr_mult` × the *current* ATR — ratcheted so it never loosens, evaluated on the close. A position exits on a close through the stop or on a confirmed start in the opposite direction. Full thesis, the five improvement runs and every caveat: [README.md](./README.md).

**The universe and windows.** `Liquid500` (497 names, ranked point-in-time on 2020-H1 dollar volume; a survivor-only pool). Data from 2020-07 (warm-up); every trial is scored **warm from 2022-01-03** to the train cutoff 2024-12-31; **2025+ is the holdout**, read once at the end (§12b).

**Flow:** load → overview → backtest + baseline → event test → inspect → ablation → optimize → read the study → boundary → stability → picked backtest → verify → validate (per-year, holdout, costs & fills, attribution & concentration) → read honestly.

## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from dataclasses import replace

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from trading_simple.config import load_settings
from trading_simple.db import make_engine
from trading_simple import backtest as bt, jobs, optimize as opt, plots, strategies
from trading_simple.data import load_panel, symbols_with_bars

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

strat = strategies.load("trend_following")
compose = strat.signals.__globals__["compose"]   # every intermediate: starts, stop line, position
d = strat.data
SCORE_FROM = strat.optuna["score_from"]          # the study scores every trial warm from here
INIT_CASH = settings.backtest.init_cash
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
print(f"{d['universe']}: {len(syms)} symbols, {panel.close.shape[0]:,} daily bars "
      f"{panel.close.index[0].date()} → {panel.close.index[-1].date()} "
      f"(scored from {SCORE_FROM}; the bars before are warm-up)")
print("fill model:", strat.fill, "| order notional:", strat.order_notional,
      "| study:", strat.optuna["study"])

## 2 · Strategy overview — hyperparameters and search ranges

| param | role in the strategy | intuition |
|---|---|---|
| `entry_window` | Donchian lookback of the start | short = early entries; the study ships 9 because relative strength already picks the names |
| `adx_window`, `adx_min` | ADX confirmation of the start (±DI must agree) | only directional breakouts count; `adx_min` 0 = off (ablation grid) |
| `atr_window`, `atr_mult` | the chandelier stop: distance in ATRs, on the current ATR, ratcheted | tighter = smaller losses, more shake-outs — every tight stop lost here |
| `direction` | long / short / both | the grid runs each side alone |
| `max_positions` | equal slots in one book (portfolio mode) | a risk decision — never searched |
| `rs_window`, `rs_skip` | 12-1 month relative strength that ranks starts competing for a slot | the literature's momentum horizon; the last month reverses |

The table below shows, per parameter: the **shipped value** (§3), the **ablation grid** (§6) and the **Optuna search range** (§7). A `—` means the stage keeps the shipped value.

In [ ]:
def _fmt_space(spec):
    # One readable line for an optuna.space entry.
    if spec["type"] == "categorical":
        return " | ".join(map(str, spec["choices"]))
    tail = " (log)" if spec.get("log") else (f", step {spec['step']}" if spec.get("step") else "")
    return f"{spec['low']} – {spec['high']}{tail}"

pd.DataFrame({
    "shipped": pd.Series(strat.params),
    "ablation grid (§6)": pd.Series({k: ", ".join(map(str, v))
                                     for k, v in strat.sweep.items()}),
    "optuna search range (§7)": pd.Series({k: _fmt_space(s)
                                           for k, s in strat.optuna["space"].items()}),
    "doc": pd.Series(strat.docs).str.slice(0, 90) + "…",
}).fillna("—")

## 3 · One backtest of the shipped config — and the baseline it has to beat

Scored **warm** from `SCORE_FROM`: the signals see the whole history (the 252-bar relative strength is defined on day one), the book starts flat on that bar. Costs are always on, priced at the strategy's real order size (`bt.strategy_costs`). The second figure is the **mandatory baseline comparison**: the equal-weight buy-and-hold of the same names over the same window (`strategies/baseline`), and `bt.evaluate` measures `excess_cagr` / `alpha_cagr` / `info_ratio` against exactly that.

> **Capital model:** ONE shared book of 20 equal slots (engine portfolio mode). Each position targets ±1/20 of the book's value at entry, drifts untouched and is sold whole.

In [ ]:
pf = bt.run_strategy(strat, panel, settings=settings, score_from=SCORE_FROM)
stats0 = bt.evaluate(strat, panel, {}, settings=settings, score_from=SCORE_FROM)
display(pd.Series(stats0).astype(float).round(4).to_frame("shipped (train, warm)"))
plots.plot_equity(bt.equity_curve(pf, init_cash=INIT_CASH),
                  title=f"{strat.name} — equity, shipped config").show()

pf_base = bt.run_strategy(strategies.load("baseline"), panel, settings=settings,
                          score_from=SCORE_FROM)
fig = go.Figure()
fig.add_trace(plots._scatter(
    bt.equity_curve(pf_base, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
    name="baseline (buy & hold)", line={"color": "#898781", "dash": "dash", "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>baseline</extra>"))
fig.add_trace(plots._scatter(
    bt.equity_curve(pf, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0,
    name=strat.name, line={"color": plots.SERIES[0], "width": 1.5},
    hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>" + strat.name + "</extra>"))
plots._base(fig, title=f"{d['universe']} · train — cumulative return vs equal-weight buy-and-hold",
            height=420)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()

## 4 · Event test — does a trend start carry information on its own?

Before any optimisation: every start at the shipped parameters, scored by its forward return from the next open (the execution bar) over 5 / 21 / 63 bars, **signed for the trade** (a short gains when the price falls), raw and **market-demeaned** (each day's cross-sectional mean removed — what is left is selection). t-statistics use monthly-clustered means, because starts cluster in time and their windows overlap.

What this study found (README): long starts carry little on their own, and **short starts are followed by a relative bounce** at one month (short-term reversal in liquid large caps). That is why the book's edge has to come from *which* starts get a slot (§6, relative strength) and from the exits, not from the start itself.

In [ ]:
HORIZONS = (5, 21, 63)
parts0 = compose(panel, **strat.params)
lo = pd.Timestamp(SCORE_FROM, tz="UTC")

def event_table(mask, sign):
    # Forward return after each start, from the next open, signed for the trade.
    rows = {}
    for h in HORIZONS:
        fwd = panel.close.shift(-h) / panel.open_.shift(-1) - 1.0
        for name, frame in (("raw", fwd), ("demeaned", fwd.sub(fwd.mean(axis=1), axis=0))):
            ev = (sign * frame).where(mask).loc[lo:].stack()
            by_month = ev.groupby(ev.index.get_level_values(0).strftime("%Y-%m")).mean()
            t = (by_month.mean() / (by_month.std() / np.sqrt(len(by_month)))
                 if len(by_month) > 2 and by_month.std() else np.nan)
            rows[f"{h}d {name}"] = {"events": len(ev), "mean": ev.mean(),
                                    "hit_rate": (ev > 0).mean(), "t (monthly)": t}
    return pd.DataFrame(rows).T

FMT = {"events": "{:.0f}", "mean": "{:+.2%}", "hit_rate": "{:.0%}", "t (monthly)": "{:+.2f}"}
for label, key, sign in (("LONG starts", "long_start", 1), ("SHORT starts", "short_start", -1)):
    print(f"\n{label}: {int(parts0[key].loc[lo:].to_numpy().sum())} events")
    display(event_table(parts0[key], sign).style.format(FMT))

## 5 · Inspect a symbol — the channel, the chandelier and the trades

`inspect_symbol` draws one name: the Donchian channel a start must close beyond (approximated from closes), the **chandelier stop** actually in force (from `compose`, NaN when flat — note how it only ever moves in the trade's favour), ▲/▼ entries and × exits. Used twice: here on the shipped config, again in §11 on the optimized pick.

In [ ]:
def inspect_symbol(sym, pf, params):
    # Deep-dive one symbol of a book: annotated chart + its trade table.
    merged = {**strat.params, **params}
    window = pf.wrapper.index
    parts = compose(panel, **merged)
    overlays = strat.indicators(panel.close[sym], panel.volume[sym], **merged)
    overlays = {**overlays, "price": {**{k: v.loc[window] for k, v in overlays["price"].items()},
                                      "chandelier stop": parts["stop"][sym].loc[window]}}
    trades = bt.trade_records(pf)
    plots.plot_symbol(panel.close[sym].loc[window], panel.volume[sym].loc[window], trades,
                      symbol=sym, overlays=overlays).show()
    mine = trades[trades["symbol"] == sym]
    return mine.sort_values("entry_ts").reset_index(drop=True).round(3)

trades0 = bt.trade_records(pf)
inspect_symbol(trades0["symbol"].value_counts().index[0], pf, {})

## 6 · Ablation grid — which side, the ADX confirmation, and how wide a stop?

Not a tuning sweep: `direction` (long / short / both) × `adx_min` (off / shipped) × `atr_mult` (tight / mid / shipped), everything else shipped — 18 combos, each scored **warm** through `bt.evaluate` exactly like a study trial (`bt.sweep` has no warm start, and this strategy's 252-bar relative strength needs one). Three questions: does either side carry the result alone? Does the ADX confirmation earn its keep? And the one the whole study kept answering — does a tight ATR stop ever pay here?

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
rows = []
for combo in grid:
    st = bt.evaluate(strat, panel, combo, settings=settings, score_from=SCORE_FROM)
    rows.append({**combo, **st})
sweep_stats = pd.DataFrame(rows)
cols = ["direction", "adx_min", "atr_mult", "excess_cagr", "alpha_cagr", "beta", "volatility",
        "max_drawdown", "pct_profitable", "trades_per_year"]
display(sweep_stats[cols].sort_values("excess_cagr", ascending=False).round(3))
plots.plot_heatmap_grid(sweep_stats, pairs=[("direction", "atr_mult"), ("adx_min", "atr_mult")],
                        metrics=("excess_cagr", "alpha_cagr", "volatility")).show()

## 7 · Optimize (resumable, runs outside the notebook)

`jobs.run_optimize` hands the work to a `tsim optimize` subprocess. The study scores **`excess_cagr` (maximized) against `volatility` (minimized)**, every trial warm from `optuna.score_from`; trials below `optuna.min_trades_per_year` are pruned in-objective. This cell only **tops the study up to `optuna.n_trials`** — re-running it never adds trials past that, so the notebook reads the same study every time. Raise `n_trials` in `config.yaml` (and keep the study name) to search deeper.

> **One worker.** Several workers on one study share the config `seed` and propose the same parameter sequence (see `momentum_swing/IDEAS.md`).

In [ ]:
path = opt.study_path(strat, strat.optuna["study"])
done = len(opt.load_study(strat).trials) if path.exists() else 0
missing = max(0, int(strat.optuna["n_trials"]) - done)
print(f"study {strat.optuna['study']}: {done} trials on disk, running {missing} more")
if missing:
    jobs.run_optimize("trend_following", n_trials=missing)

## 8 · Read the study — the pick

The repo's one picking rule (`opt.pick_from_front`): among the Pareto front, only rows that **beat buy-and-hold** are eligible, reduced to one by the balanced `opt.PICK_WEIGHTS` after collapsing ties. The in-objective trade floor already made every scored trial admissible.

In [ ]:
study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)
space = strat.optuna["space"]
_INTS = {k for k, s in space.items() if s["type"] == "int"}

def coerce(row):
    # Cast a front row back to real parameter types (front rows store floats).
    return {k: (int(round(float(row[k]))) if k in _INTS else float(row[k])) for k in space}

best_row = opt.pick_from_front(front)
picked = coerce(best_row)
print(f"trials: {len(study.trials)} total, {len(everything)} scored, "
      f"{int((everything['excess_cagr'] > 0).sum())} beat buy-and-hold; "
      f"front: {len(front)} rows, {int(best_row['front_distinct'])} distinct outcomes")
print("picked params:", picked, "| shipped in config.yaml:",
      {k: strat.params[k] for k in space} == picked)
print("picked train metrics:", {m: round(float(best_row[m]), 4) for m in
      ("excess_cagr", "volatility", "alpha_cagr", "beta", "info_ratio", "max_drawdown",
       "trades_per_year")})
display(opt.distinct_front(front)[["trial", *space, "excess_cagr", "volatility", "alpha_cagr",
                                   "beta", "trades_per_year"]].round(4))
plots.plot_pareto(front, everything, picked=int(best_row["trial"])).show()

## 9 · Boundary check — did the optimum hit the search-space edge?

`optimize.boundary_report` flags any picked parameter within 5% of a searched bound (log-aware), and any bound half the distinct front piles onto. If something is flagged: widen that bound in `config.yaml`, **bump `optuna.study`**, re-run §7 and re-read §8–§9.

In [ ]:
report = opt.boundary_report(picked, space, front=front)
flagged = report[report["action"] != "ok"]
print("all clear — the searched ranges bracket the optimum" if flagged.empty
      else f"{len(flagged)} parameter(s) pressing against the search space:")
report

## 10 · Stability — plateau or spike?

The study reports its best point; it says nothing about whether that point is a **plateau** or a **spike**. This strategy's landscape is spiky by construction — with a full book, one different exit changes which names take the next slots — so every picked value is perturbed by ±5% and ±10% of its searched span, one at a time, and each neighbour is scored warm on the train window. Read the tornado (how far each knob moves the result) and the verdicts: a **spike** is the overfit case — prefer a flatter front row.

In [ ]:
star = opt.neighbors(picked, space, valid=lambda p: strat.valid({**strat.params, **p}))
rows = []
for rec in star.to_dict("records"):
    p = {k: (int(rec[k]) if k in _INTS else float(rec[k])) for k in space}
    st = bt.evaluate(strat, panel, p, settings=settings, score_from=SCORE_FROM)
    rows.append({**p, "perturbed": rec["perturbed"], "offset": rec["offset"], **st})
star = pd.DataFrame(rows)
moved = star[star["perturbed"] != ""]
print(f"{len(moved)} neighbours: {int((moved['excess_cagr'] > 0).sum())} beat buy-and-hold, "
      f"median excess {moved['excess_cagr'].median():+.2%}/yr "
      f"(centre {star.loc[star['perturbed'] == '', 'excess_cagr'].iloc[0]:+.2%})")
display(opt.stability_report(star))
plots.plot_sensitivity(star).show()

## 11 · Backtest the picked parameters — and verify them symbol by symbol

In [ ]:
pf_best = bt.run_strategy(strat, panel, params=picked, settings=settings, score_from=SCORE_FROM)
stats_best = bt.evaluate(strat, panel, picked, settings=settings, score_from=SCORE_FROM)
display(pd.Series(stats_best).astype(float).round(4).to_frame("picked (train, warm)"))
plots.plot_equity(bt.equity_curve(pf_best, init_cash=INIT_CASH),
                  title=f"{strat.name} — equity with picked parameters").show()
# A symbol's total_return here is its CONTRIBUTION to the one book, not a standalone return.
display(bt.symbol_stats(pf_best).round(4).head(15))

In [ ]:
SYMBOL = None   # ← set to any symbol from the table above; None = the biggest contributor
sym = SYMBOL or bt.symbol_stats(pf_best).index[0]
inspect_symbol(sym, pf_best, picked)

## 12 · Validate — is the edge real?

Everything above is in-sample. Every window below is scored **warm**: `bt.evaluate(strat, <long panel>, picked, score_from=<window start>)` — the signals run on the whole history, the book and the benchmark start on the window's first bar. Never a panel sliced to the window: that would restart the 252-bar relative strength inside it.

### 12a · The hurdle, year by year

Three years can hide a strategy that worked in one of them — and a long/short trend follower is expected to earn its keep in the bear market. Each calendar year is its own warm-scored book against its own buy-and-hold.

In [ ]:
full = load_panel(engine, syms, d["bar"], d["start"], None,
                  session=d.get("session", "regular"), exchanges=settings.exchanges)
HOLDOUT_START = "2025-01-02"
LAST = str(full.close.index[-1].date())
print(f"full history: {full.close.shape[0]:,} bars {full.close.index[0].date()} → {LAST}")

def upto(p, end):
    # The panel up to `end` (inclusive) — used only to END a window; its start stays warm.
    ts = pd.Timestamp(end, tz="UTC")

    def cut(frame):
        return frame.loc[frame.index <= ts]

    return replace(p, close=cut(p.close), open_=cut(p.open_), high=cut(p.high),
                   low=cut(p.low), volume=cut(p.volume))

KEEP = ("cagr", "excess_cagr", "alpha_cagr", "beta", "volatility", "max_drawdown",
        "info_ratio", "trades_per_year")
years = {}
for label, start, end in (("train 2022-24", SCORE_FROM, "2024-12-31"),
                          ("2022", "2022-01-03", "2022-12-31"),
                          ("2023", "2023-01-03", "2023-12-31"),
                          ("2024", "2024-01-02", "2024-12-31")):
    st = bt.evaluate(strat, upto(full, end), picked, settings=settings, score_from=start)
    years[label] = {"buy_hold_cagr": st["cagr"] - st["excess_cagr"], **{k: st[k] for k in KEEP}}
display(pd.DataFrame(years).T.astype(float).round(3))

### 12b · The untouched holdout (open once)

2025+ was never seen by any study, grid or design decision. **Pass** = holdout `excess_cagr` > 0 **and** `info_ratio` > 0; read `volatility` beside buy-and-hold's (the study minimized it) and `alpha_cagr` beside `beta` (this book's beta swings with the market's direction). If it disappoints, that is information — do not tune against it.

In [ ]:
st_hold = bt.evaluate(strat, full, picked, settings=settings, score_from=HOLDOUT_START)
bh_hold = bt.evaluate(strategies.load("baseline"), full, {}, settings=settings,
                      score_from=HOLDOUT_START)
print(f"HOLDOUT {HOLDOUT_START} → {LAST} ({st_hold['years']:.2f} years, "
      f"{st_hold['trades_per_year']:.0f} trades/yr)")
print(f"  strategy     {st_hold['cagr']:+7.2%}/yr   volatility {st_hold['volatility']:.1%}   "
      f"max drawdown {st_hold['max_drawdown']:.1%}")
print(f"  buy & hold   {st_hold['cagr'] - st_hold['excess_cagr']:+7.2%}/yr   "
      f"volatility {bh_hold['volatility']:.1%}   max drawdown {bh_hold['max_drawdown']:.1%}")
print(f"  EXCESS       {st_hold['excess_cagr']:+7.2%}/yr   information ratio "
      f"{st_hold['info_ratio']:+5.2f}   alpha {st_hold['alpha_cagr']:+.2%} "
      f"at beta {st_hold['beta']:+.2f}")
ok = st_hold["excess_cagr"] > 0 and st_hold["info_ratio"] > 0
print("  -->", "PASS — it beat owning the universe out of sample" if ok
      else "FAIL — owning the universe was better out of sample")
for label, start, end in (("2025", "2025-01-02", "2025-12-31"), ("2026", "2026-01-02", LAST)):
    st = bt.evaluate(strat, upto(full, end), picked, settings=settings, score_from=start)
    print(f"  {label}: excess {st['excess_cagr']:+.2%}  alpha {st['alpha_cagr']:+.2%}  "
          f"beta {st['beta']:+.2f}  max drawdown {st['max_drawdown']:.1%}")

pf_hold = bt.run_strategy(strat, full, params=picked, settings=settings, score_from=HOLDOUT_START)
pf_bh = bt.run_strategy(strategies.load("baseline"), full, settings=settings,
                        score_from=HOLDOUT_START)
fig = go.Figure()
for p, name, color, dash in ((pf_bh, "baseline (buy & hold)", "#898781", "dash"),
                             (pf_hold, strat.name, plots.SERIES[0], None)):
    curve = bt.equity_curve(p, init_cash=INIT_CASH)["value"] / INIT_CASH - 1.0
    fig.add_trace(plots._scatter(
        curve, name=name, line={"color": color, "dash": dash, "width": 1.5},
        hovertemplate="%{x|%Y-%m-%d}<br>%{y:+.1%}<extra>" + name + "</extra>"))
plots._base(fig, title="HOLDOUT 2025+ — cumulative return vs equal-weight buy-and-hold", height=420)
fig.update_yaxes(tickformat="+.0%", title_text="cumulative return")
fig.show()

### 12c · Costs and fills — how much of the edge lives in the execution assumptions?

Friction at the repo's two fixed bars — **6/5** and **10/3** bps (spread/slippage per leg; `active_stats` re-prices the benchmark at the same costs) — then **pessimistic fills** (every buy at the bar's high, every sell at its low; the benchmark fills the same way).

In [ ]:
def with_costs(**overrides):
    # A copy of settings with its (pydantic) cost block overridden.
    return settings.model_copy(update={"costs": settings.costs.model_copy(update=overrides)})

rows = {}
for label, s, st_ in (("base costs", settings, strat),
                      ("6/5 bps", with_costs(spread_bps=6.0, slippage_bps=5.0), strat),
                      ("10/3 bps", with_costs(spread_bps=10.0, slippage_bps=3.0), strat),
                      ("pessimistic fills", settings, replace(strat, fill="pessimistic"))):
    for window, p, start in (("train", upto(full, "2024-12-31"), SCORE_FROM),
                             ("holdout", full, HOLDOUT_START)):
        st = bt.evaluate(st_, p, picked, settings=s, score_from=start)
        rows[(label, window)] = {"excess_cagr": st["excess_cagr"], "alpha_cagr": st["alpha_cagr"]}
pd.DataFrame(rows).T.unstack().round(4)

### 12d · Where the money came from — long vs short, and how many names carry it

Trend following lives on a few outliers, which is also why one holdout cannot confirm it. The long/short split per window, the top contributors' share of the P&L, and the holdout re-run **without its single best name** in the universe.

In [ ]:
def attribution(p, start):
    tr = bt.trade_records(bt.run_strategy(strat, p, params=picked, settings=settings,
                                          score_from=start))
    by_side = tr.groupby("direction")["pnl"].sum() / INIT_CASH
    by_name = tr.groupby("symbol")["pnl"].sum().sort_values(ascending=False)
    return by_side, by_name

for label, p, start in (("train 2022-24", upto(full, "2024-12-31"), SCORE_FROM),
                        ("holdout", full, HOLDOUT_START)):
    by_side, by_name = attribution(p, start)
    print(f"{label}: " + ", ".join(f"{k} {v:+.1%}" for k, v in by_side.items())
          + f" of the book | top-5 names = {by_name.head(5).sum() / by_name.sum():.0%} of the P&L: "
          + ", ".join(f"{k} {v / INIT_CASH:+.1%}" for k, v in by_name.head(5).items()))

best = attribution(full, HOLDOUT_START)[1].index[0]
keep = [s for s in full.symbols if s != best]
without = replace(full, close=full.close[keep], open_=full.open_[keep], high=full.high[keep],
                  low=full.low[keep], volume=full.volume[keep])
st = bt.evaluate(strat, without, picked, settings=settings, score_from=HOLDOUT_START)
print(f"holdout WITHOUT {best} in the universe: excess {st['excess_cagr']:+.2%}/yr, "
      f"alpha {st['alpha_cagr']:+.2%}")

## Read the result honestly

* **§4 says the start alone carries little**, and short starts are followed by a relative *bounce*. The book's edge comes from *which* starts get a slot (12-1 month relative strength) and from letting winners run — not from the breakout itself.
* **The ATR stop ended up a disaster stop.** Every tight chandelier lost (§6 and README: 2-5 ATRs cost 7-22 points of excess a year); the study ships 14 ATRs, and most exits are opposite starts. The trailing-stop idea is sound engineering here — it just is not where the money is on liquid US large caps.
* **Crisis alpha, not steady outperformance.** Read §12a before §12b: the train excess is 2022, the bear year; 2023 and 2024 lost to buy-and-hold. Beta swings with the market (negative in train), so `excess_cagr` largely measures which way the market went — read `alpha_cagr` beside it.
* **One holdout, one outlier.** §12d: remove the holdout's best name and the out-of-sample excess turns negative. §12c: most of the edge needs open-price fills. The next honest step is a rolling walk-forward (IDEAS.md #1), not more tuning.
* **The pool is survivor-only** — the best shorts (names that later delisted) are missing, so the short side is understated and the long side flattered.